# Skills + Flow con dos Crews

Clasificación: **Skill-augmented agents con Flow multi-crew.** Un Flow orquesta dos crews independientes: una que planifica el viaje y otra que lo audita con una skill inyectada.

Este patrón combina dos ideas:
1. **Skills**: el revisor tiene expertise inyectada via `SKILL.md` con la metodología de auditoría.
2. **Flow multi-crew**: el `@start` lanza la crew de planificación, el `@listen` lanza la crew de revisión con el resultado de la primera.

Separar en dos crews tiene sentido cuando cada una tiene agentes, tools o configuración distintos. El planificador usa la crew de viajes con tools de búsqueda; el revisor es una crew aparte, más simple, con la skill como único input extra.

## Skills vs Knowledge vs Tools

|  | Skills | Knowledge | Tools |
|---|---|---|---|
| Qué son | Instrucciones en `SKILL.md` | Datos en vectorstore (RAG) | Funciones ejecutables |
| Cómo llegan al agente | Se inyectan enteras en el prompt | Chunks relevantes via búsqueda semántica | Se invocan bajo demanda |
| Para qué | Metodología, checklists, criterios | Datos estáticos (docs, guías) | Acciones dinámicas (buscar, calcular) |

In [ ]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## Cómo funciona una skill

Una skill es un directorio con un `SKILL.md` que contiene instrucciones en markdown. Se asigna al agente con `skills=["./path"]` y CrewAI inyecta el contenido entero en su prompt.

```
skills/
└── revision-itinerarios/
    └── SKILL.md
```

### Contenido de `skills/revision-itinerarios/SKILL.md`

In [3]:
%pycat skills/revision-itinerarios/SKILL.md

---
name: revision-itinerarios
description: Metodología de revisión de itinerarios de viaje. Define los criterios de evaluación, el proceso de auditoría y el formato de reporte para validar que un itinerario es viable, coherente y respeta las restricciones del cliente.
metadata:
  author: escuderx
  version: "1.0"
---

# Revisor de Itinerarios de Viaje

Eres un auditor de itinerarios. Tu trabajo es revisar propuestas de viaje y detectar problemas antes de que el cliente los sufra.

## Proceso de revisión

Para cada itinerario, ejecuta estas verificaciones en orden:

### 1. Coherencia logística
- ¿Las distancias diarias son realistas? Máximo 300 km/día en carreteras secundarias, 500 km en autopista.
- ¿Hay tiempo suficiente entre paradas? Mínimo 1h por parada turística, 30 min por parada de descanso.
- ¿El primer día respeta la hora de llegada del vuelo? Si llegan de noche, la primera actividad es al día siguiente.
- ¿El último día deja margen para devolver coche/llegar al aeropuerto? M

## Modelo Pydantic para el itinerario

Definir un `BaseModel` como `output_pydantic` de la task fuerza al agente a devolver datos estructurados. El revisor trabaja con campos parseables en vez de texto libre. CrewAI valida la salida contra el modelo y reintenta si no encaja.

El modelo está en `models.py` para reutilizarlo desde cualquier notebook.

In [4]:
%pycat models.py

from pydantic import BaseModel
from typing import Optional


class Actividad(BaseModel):
    nombre: str
    duracion_horas: float
    coste_eur: float
    ubicacion: str


class Desplazamiento(BaseModel):
    origen: str
    destino: str
    modo: str
    distancia_km: Optional[float] = None
    duracion_min: Optional[int] = None
    coste_eur: float


class DiaItinerario(BaseModel):
    dia: int
    actividades: list[Actividad]
    desplazamientos: list[Desplazamiento]
    alojamiento: str
    coste_dia_eur: float


class Itinerario(BaseModel):
    destino: str
    dias: int
    personas: int
    vuelos_coste_eur: float
    alojamiento_coste_eur: float
    transporte_coste_eur: float
    actividades_coste_eur: float
    coste_total_eur: float
    presupuesto_eur: float
    plan: list[DiaItinerario]


In [5]:
from viajes_crew import ViajesCrew
from crewai import Agent, Task, Crew
from pydantic import BaseModel
from crewai.flow.flow import Flow, start, listen
from models import Itinerario


class ViajeSkillState(BaseModel):
    destino: str = ""
    dias: int = 0
    personas: int = 0
    presupuesto: int = 0
    itinerario_raw: str = ""
    revision: str = ""


class ViajesConRevisionFlow(Flow[ViajeSkillState]):

    @start()
    def pedir_datos(self):
        print("\n=== Planificador de Viajes con Revisión ===\n")
        self.state.destino = input("Destino: ")
        self.state.dias = int(input("Días: "))
        self.state.personas = int(input("Personas: "))
        self.state.presupuesto = int(input("Presupuesto (EUR): "))
        print(f"\nPlanificando viaje a {self.state.destino}...\n")
        return self.state

    @listen(pedir_datos)
    def planificar(self, state):
        """Crew 1: planificación con output estructurado."""
        planificador = Agent(
            role="Planificador de Viajes",
            goal=f"Crear un itinerario completo a {state.destino} dentro de {state.presupuesto} EUR",
            backstory="Diseñas viajes completos con vuelos, alojamiento, transporte y actividades.",
        )
        planificar_task = Task(
            description=(
                f"Crea un itinerario de {state.dias} días a {state.destino} para {state.personas} personas.\n"
                f"Presupuesto máximo: {state.presupuesto} EUR.\n"
                "Incluye: vuelos, alojamiento, transporte entre puntos, actividades dia a dia.\n"
                "Indica el coste de cada partida."
            ),
            expected_output="Itinerario estructurado con coste por partida y total.",
            agent=planificador,
            output_pydantic=Itinerario,
        )
        result = Crew(agents=[planificador], tasks=[planificar_task], verbose=True).kickoff()
        self.state.itinerario_raw = result.raw
        print("\n=== Itinerario generado. Pasando a revisión... ===\n")
        return result.raw

    @listen(planificar)
    def revisar(self, itinerario):
        """Crew 2: revisión con skill inyectada."""
        revisor = Agent(
            role="Revisor de Itinerarios",
            goal="Auditar itinerarios verificando logística, presupuesto y riesgos",
            backstory="Auditor de viajes. No generas itinerarios, los revisas.",
            skills=["./skills/revision-itinerarios"],
        )
        revisar_task = Task(
            description=(
                f"Revisa este itinerario:\n\n{itinerario}\n\n"
                f"Presupuesto máximo: {self.state.presupuesto} EUR.\n"
                "Genera el reporte con el formato definido en tu skill."
            ),
            expected_output="Reporte de revisión con veredicto, puntuación y acciones requeridas.",
            agent=revisor,
        )
        result = Crew(agents=[revisor], tasks=[revisar_task], verbose=True).kickoff()
        self.state.revision = result.raw
        return result.raw

## Ejecución

El flow pide datos, lanza la crew de planificación y luego la crew de revisión.

In [6]:
flow = ViajesConRevisionFlow()
result = flow.kickoff()
print(result)

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: ViajesConRevisionFlow                                                                                    │
│  ID: fbdbe16a-30f2-4559-96e3-4ba4a688044b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: ViajesConRevisionFlow                                                                                    │
│  ID: fbdbe16a-30f2-4559-96e3-4ba4a688044b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: fbdbe16a-30f2-4559-96e3-4ba4a688044b

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: pedir_datos                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


=== Planificador de Viajes con Revisión ===



...

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: bd1e66d3-0557-48df-a461-82e1c6fcf4bf                                                                       │
│  Final Output: Reporte de Revisión de Itinerario - Viaje a Islandia (12 días, 4 personas)                       │
│                                                                                                                 │
│  1. Aspectos Logísticos:                                                                                        │
│                                                                                                                 │
│  - Transporte:                                                                                                  │
│    • Se contemplan vuelos internacionales por 1.600 EUR, alquiler de coche cubriendo la mayor parte de          │
│  desplazamientos en Islandia, además de shuttle bus y tours organizados.                                        │
│    • El itinerario detalla correctamente distancias y tiempos en km y minutos, y los medios de transporte       │
│  elegidos para cada tramo parecen adecuados para optimizar tiempos y experiencias.                              │
│    • El último día se contempla vuelo interno más shuttle para regreso al aeropuerto, con coste de 120 EUR      │
│  incluido.                                                                                                      │
│  - Alojamiento:                                                                                                 │
│    • Se presentan alojamientos variados (hoteles 3 estrellas, guesthouses) en los lugares clave del recorrido.  │
│    • El detalle de costes por día y lugar parece coherente con los estándares de Islandia.                      │
│  - Actividades y Duración:                                                                                      │
│    • Actividades correctamente distribuidas en el tiempo con estimaciones realistas de duración.                │
│    • Gran variedad de experiencias emblemáticas de Islandia (Golden Circle, Laguna Azul, cascadas, Parque       │
│  Nacional, laguna glaciar, fiordos, avistamiento de ballenas).                                                  │
│  - Riesgos y Recomendaciones Logísticas:                                                                        │
│    • Las distancias diarias más largas (como el día 8: 250km y día 10: 260 km) implican muchas horas de         │
│  conducción; se recomienda prever descansos para evitar fatiga.                                                 │
│    • Importante verificar condiciones climáticas locales para conducción en fiordos y zonas remotas.            │
│    • Se sugiere asegurarse de que el alquiler de coche contempla cobertura adecuada ante condiciones            │
│  islandesas (nieve, hielo).                                                                                     │
│                                                                                                                 │
│  2. Aspectos Presupuestarios:                                                                                   │
│                                                                                                                 │
│  - Costes desglosados:                                                                                          │
│    • Vuelos internacionales: 1.600 EUR                                                                          │
│    • Alojamiento total: 2.000 EUR (media ~167 EUR/día)

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: revisar                                                                                                │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: ViajesConRevisionFlow                                                                                    │
│  ID: fbdbe16a-30f2-4559-96e3-4ba4a688044b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 8d506b23-f898-44ff-ab3d-e2b3f58014eb                                  │
│                                                                                                                 │
│ 🔗 View here: https://app.crewai.com/crewai_plus/trace_batches/8d506b23-f898-44ff-ab3d-e2b3f58014eb             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Reporte de Revisión de Itinerario - Viaje a Islandia (12 días, 4 personas)

1. Aspectos Logísticos:

- Transporte:
  • Se contemplan vuelos internacionales por 1.600 EUR, alquiler de coche cubriendo la mayor parte de desplazamientos en Islandia, además de shuttle bus y tours organizados.
  • El itinerario detalla correctamente distancias y tiempos en km y minutos, y los medios de transporte elegidos para cada tramo parecen adecuados para optimizar tiempos y experiencias.
  • El último día se contempla vuelo interno más shuttle para regreso al aeropuerto, con coste de 120 EUR incluido.
- Alojamiento:
  • Se presentan alojamientos variados (hoteles 3 estrellas, guesthouses) en los lugares clave del recorrido.
  • El detalle de costes por día y lugar parece coherente con los estándares de Islandia.
- Actividades y Duración:
  • Actividades correctamente distribuidas en el tiempo con estimaciones realistas de duración.
  • Gran variedad de experiencias emblemáticas de Islandia (Golden Circ

## Presentación del itinerario

In [7]:
import json
from IPython.display import display, Markdown

itinerario_data = json.loads(flow.state.itinerario_raw)

resumen = f"""## {itinerario_data['destino']} - {itinerario_data['dias']} días, {itinerario_data['personas']} personas

| Partida | Coste |
|---------|-------|
| Vuelos | {itinerario_data['vuelos_coste_eur']} EUR |
| Alojamiento | {itinerario_data['alojamiento_coste_eur']} EUR |
| Transporte | {itinerario_data['transporte_coste_eur']} EUR |
| Actividades | {itinerario_data['actividades_coste_eur']} EUR |
| **Total** | **{itinerario_data['coste_total_eur']} EUR** |
| Presupuesto | {itinerario_data['presupuesto_eur']} EUR |

---
"""

for dia in itinerario_data["plan"]:
    resumen += f"\n### Día {dia['dia']} - {dia['alojamiento']} ({dia['coste_dia_eur']} EUR)\n\n"
    resumen += "| Actividad | Ubicación | Duración | Coste |\n"
    resumen += "|-----------|-----------|----------|-------|\n"
    for act in dia["actividades"]:
        resumen += f"| {act['nombre']} | {act['ubicacion']} | {act['duracion_horas']}h | {act['coste_eur']} EUR |\n"
    if dia["desplazamientos"]:
        resumen += "\n| Desplazamiento | Modo | Distancia | Tiempo | Coste |\n"
        resumen += "|----------------|------|-----------|--------|-------|\n"
        for d in dia["desplazamientos"]:
            dist = f"{d['distancia_km']} km" if d.get('distancia_km') else "-"
            dur = f"{d['duracion_min']} min" if d.get('duracion_min') else "-"
            resumen += f"| {d['origen']} → {d['destino']} | {d['modo']} | {dist} | {dur} | {d['coste_eur']} EUR |\n"

display(Markdown(resumen))

## Islandia - 12 días, 4 personas

| Partida | Coste |
|---------|-------|
| Vuelos | 1600.0 EUR |
| Alojamiento | 2000.0 EUR |
| Transporte | 800.0 EUR |
| Actividades | 1400.0 EUR |
| **Total** | **5800.0 EUR** |
| Presupuesto | 6000.0 EUR |

---

### Día 1 - Hotel 3 estrellas en Reikiavik (260.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Llegada a Reikiavik y check-in en el alojamiento | Reikiavik | 2.0h | 0.0 EUR |
| Paseo por el centro de Reikiavik | Reikiavik | 3.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Aeropuerto Keflavik → Hotel en Reikiavik | Shuttle bus | 50.0 km | 45 min | 80.0 EUR |

### Día 2 - Hotel 3 estrellas en Reikiavik (320.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Tour Golden Circle (Parque Thingvellir, Geysir, Gullfoss) | Golden Circle | 8.0h | 140.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Reikiavik → Golden Circle | Tour organizado en autobús | 300.0 km | 480 min | 0.0 EUR |

### Día 3 - Alojamiento cercano a Laguna Azul (320.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Visita a la Laguna Azul | Laguna Azul | 4.0h | 160.0 EUR |
| Relajación y spa en Laguna Azul | Laguna Azul | 2.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Reikiavik → Laguna Azul | Coche alquilado | 50.0 km | 50 min | 0.0 EUR |

### Día 4 - Guesthouse en Selfoss (180.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Desplazamiento en coche hacia el Círculo Dorado Sur | Sureste de Islandia | 4.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Laguna Azul → Selfoss | Coche alquilado | 70.0 km | 70 min | 0.0 EUR |

### Día 5 - Guesthouse en Skogar (200.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Visita a la Cascada Seljalandsfoss y Skogafoss | Cascadas del sur | 5.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Selfoss → Skogafoss | Coche alquilado | 110.0 km | 90 min | 0.0 EUR |

### Día 6 - Guesthouse en Skaftafell (220.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Senderismo en el Parque Nacional Skaftafell | Skaftafell | 6.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Skogar → Skaftafell | Coche alquilado | 150.0 km | 120 min | 0.0 EUR |

### Día 7 - Guesthouse en Höfn (260.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Tour en barco por la laguna glaciar Jökulsárlón | Jökulsárlón | 2.0h | 120.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Skaftafell → Jökulsárlón | Coche alquilado | 60.0 km | 50 min | 0.0 EUR |

### Día 8 - Guesthouse en Egilsstaðir (180.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Conducción hacia Egilsstaðir, visitando fiordos del este | Fiordos del este | 6.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Höfn → Egilsstaðir | Coche alquilado | 250.0 km | 180 min | 0.0 EUR |

### Día 9 - Guesthouse en Egilsstaðir (180.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Exploración de Seydisfjordur | Seydisfjordur | 4.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Egilsstaðir → Seydisfjordur | Coche alquilado | 27.0 km | 30 min | 0.0 EUR |

### Día 10 - Hotel en Akureyri (250.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Conducción hacia Akureyri, con paradas escénicas | Norte de Islandia | 7.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Egilsstaðir → Akureyri | Coche alquilado | 260.0 km | 190 min | 0.0 EUR |

### Día 11 - Hotel en Akureyri (310.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Whale watching en Akureyri | Akureyri | 3.0h | 160.0 EUR |

### Día 12 - No aplica (vuelo de salida) (120.0 EUR)

| Actividad | Ubicación | Duración | Coste |
|-----------|-----------|----------|-------|
| Regreso a Reikiavik | Itinerario Reikiavik | 5.0h | 0.0 EUR |

| Desplazamiento | Modo | Distancia | Tiempo | Coste |
|----------------|------|-----------|--------|-------|
| Akureyri → Aeropuerto Keflavik | Vuelo interno + shuttle | - | 120 min | 120.0 EUR |
